# 22 · Transformer 编码层

> **本节属于 Part 8 · 注意力与 Transformer。**

注意力、多头、位置编码都备齐了。本节把它们和**前馈网络 (FFN)**、**残差连接**、**LayerNorm** 组装成一个完整的 **Transformer 编码层**——这正是 BERT、GPT 等大模型的基本积木。

## 学习目标

- 理解 Transformer 编码层的结构：**多头注意力子层 + 前馈子层**，各带**残差 + LayerNorm**
- 理解残差连接（让梯度顺畅流动）、FFN（逐位置的非线性变换）、LayerNorm（稳定训练）各自的作用
- 用 `Embedding + 位置编码 + 多个编码层` 搭出一个编码器

## 一个编码层 = 两个子层

$$\text{子层1：}\quad x \leftarrow x + \text{MultiHeadAttn}(\text{LN}(x))$$
$$\text{子层2：}\quad x \leftarrow x + \text{FFN}(\text{LN}(x))$$

- **残差连接** $x + \text{Sublayer}(x)$：给梯度一条"近路"，让很深的网络也能训练（类似 ResNet）。
- **FFN**：对每个位置独立做 `Linear → ReLU → Linear`，提供逐位置的非线性变换能力。
- **LayerNorm**：稳定每层的激活分布（我们用 **pre-norm**：LN 放在子层之前，更易训练）。

In [ ]:
import inspect
import numpy as np
import minitorch
from minitorch import Tensor, nn

print(inspect.getsource(nn.TransformerEncoderLayer.forward))

## 搭一个编码器

`Embedding`（token → 向量）+ 位置编码 + 若干编码层，就是一个 Transformer 编码器。

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab, d_model=32, heads=4, d_ff=64, layers=2):
        super().__init__()
        self.emb = nn.Embedding(vocab, d_model)
        self.pos = nn.PositionalEncoding(d_model)
        self.layers = [nn.TransformerEncoderLayer(d_model, heads, d_ff) for _ in range(layers)]
    def forward(self, idx, mask=None):
        x = self.pos(self.emb(idx))
        for layer in self.layers:
            x = layer(x, mask)
        return x

minitorch.set_seed(0)
enc = Encoder(vocab=20, layers=2)
idx = np.random.randint(0, 20, size=(2, 7))     # (N, L) 的 token id
out = enc(idx)
print("输入 token (2,7) -> 编码输出", out.shape)
print("可学习参数张量个数:", len(enc.parameters()))

## 验证：梯度能流过整个深层堆叠

残差连接的意义在于让梯度顺畅回流。我们对输出求和反向，确认 Embedding 也收到了非零梯度。

In [ ]:
out.sum().backward()
print("Embedding 收到梯度（非零）:", np.abs(enc.emb.weight.grad).sum() > 0)
print("第一层注意力权重 Wq 收到梯度:", np.abs(enc.layers[0].attn.Wq.weight.grad).sum() > 0)

## PyTorch 对照

`nn.TransformerEncoderLayer` 是同样的结构（PyTorch 默认 post-norm，可切换 `norm_first=True` 变成我们用的 pre-norm）。

In [ ]:
import torch
tlayer = torch.nn.TransformerEncoderLayer(d_model=32, nhead=4, dim_feedforward=64,
                                          batch_first=True, norm_first=True)
x = torch.randn(2, 7, 32)
print("PyTorch 编码层输出:", tuple(tlayer(x).shape), " （与我们的 (2,7,32) 一致）")

## 📦 沉淀进 minitorch

`TransformerEncoderLayer` 在 `minitorch/nn/transformer.py`，`Embedding` 在 `minitorch/nn/embedding.py`，由 `tests/test_transformer.py` 守护。

## 小练习

1. **去掉残差**：把编码层里的 `x +` 残差去掉，叠很多层后还能训练吗？观察梯度是否变小。
2. **post-norm vs pre-norm**：把 LN 改成放在子层**之后**（post-norm），比较训练稳定性。
3. **FFN 宽度**：把 `d_ff` 从 `2*d_model` 调到 `4*d_model`（原论文比例），参数量和表达力如何变化？

## 小结 & 下一站

✅ 我们组装出了完整的 Transformer 编码层（注意力 + FFN + 残差 + LayerNorm），并堆叠成编码器，确认梯度能流过整个深层结构。

**下一站 → `23_train_tiny_transformer`（全项目高潮）**：用这个 Transformer **真刀真枪训练**一个序列任务，并可视化它学到的注意力模式——你会看到注意力清晰地"对齐"到正确的位置！